# Agentic Energy OS -- Henkel Düsseldorf Holthausen Pilot
### Data-Driven Energy Business Use Cases & Mathematical Optimization Framework (EUR/ton)

**Prepared for:** RIZM Application Challenge  
**Target Site:** Henkel Flagship Chemical & Consumer Goods Site, Düsseldorf-Holthausen  
**Primary Metric:** EUR / ton of industrial output  
**Methodology:** MILP Optimization via `oemof.solph` & `HiGHS` + Thermodynamic Dual-Temperature Quality + Real 2025 SMARD Wholesale Market Data  

---

## Executive Summary & Methodology First Approach

Henkel’s Düsseldorf-Holthausen site is one of the largest integrated chemical and consumer goods production complexes in Europe (~450,000 tons/year combined output spanning laundry detergents, home care, and industrial adhesives). Energy costs directly dictate site competitiveness.

This deliverable demonstrates how RIZM’s **Agentic Energy OS** unlocks immediate operational margin (Operation Hub) and optimizes capital expenditure (Decision Hub) while strictly respecting physical exergy constraints and German energy market regulations (sec 19 Abs. 2 StromNEV).

### Key Methodology Choices:
1. **Dual Thermal Quality Streams:** High-Temperature Steam (16 bar, ~200 deg C) served by CHP / Gas Boilers vs Mid/Low-Temperature Process Heat (~80 deg C) served by High-Temperature Industrial Heat Pumps (HTHP, COP 2.8).
2. **Real 2025 German Market Data:** SMARD Day-Ahead electricity prices (filter 4169, DE bidding zone) combined with Open-Meteo Düsseldorf weather profiles.
3. **Co-Optimization of CAPEX & OPEX:** Annualized Equivalent Costs (EAC) used in `oemof.solph` Investment mode to size Rooftop Solar PV, Battery Storage (BESS), Heat Pumps, and Thermal Storage (TES).


In [1]:
import sys
import importlib
from pathlib import Path
import pandas as pd
from IPython.display import display

# Ensure latest module state is loaded in interactive Jupyter sessions
import src.utils
import src.optimization_model
importlib.reload(src.utils)
importlib.reload(src.optimization_model)

# Import data preparation, Pydantic schemas, and visualization abstractions
from src.external_api import prepare_data_files
from src.optimization_model import (
    HenkelEnergySystem,
    FacilityProjectConfig,
    FixedSizingConfig,
    VariableSizingConfig,
)
from src.utils import (
    plot_dispatch_stacks,
    plot_dispatch_stacks_interactive,
    plot_storage_dynamics_interactive,
    plot_price_duration_curves_interactive,
    plot_asset_economics_interactive,
    plot_sec19_grid_fee_protection_interactive,
    create_summary_dataframe,
    create_pypsa_asset_sizing_table,
)

print("Core PyPSA Modules, Pydantic Schemas & PyPSA Reporting Suite Loaded Successfully.")


Core PyPSA Modules, Pydantic Schemas & PyPSA Reporting Suite Loaded Successfully.


## 1. Baseline Fermi Estimate & EUR/ton Cost Derivation

To establish a defensible baseline before running complex MILP models, we ground Henkel Holthausen's energy demands in published industrial studies (*StoREN Phase 1 DLR/Henkel report* and *Bolten et al. 2026*):

- **Continuous Electrical Load ($P_{el}$):** ~60 MW_el
- **Continuous Thermal Load ($P_{th}$):** ~220 MW_th (split into 160 MW_th High-Temp Steam & 60 MW_th Mid-Temp Process Heat)
- **Operating Hours:** 7,000 full-load hours/year (~80% capacity factor)
- **Annual Industrial Output Baseline:** 450,000 tons/year

### Baseline Energy Intensity Derivation:
- **Thermal Energy Intensity:** $\frac{220\text{ MW} \times 7,000\text{ h}}{450,000\text{ tons}} = 3.422\text{ MWh}_{th} / \text{ton}$
- **Electrical Energy Intensity:** $\frac{60\text{ MW} \times 7,000\text{ h}}{450,000\text{ tons}} = 0.933\text{ MWh}_{el} / \text{ton}$

### Baseline Energy Tariffs (2025 Benchmark):
- **Weighted Natural Gas + CO2 Tax (EUR 85/t):** EUR 59.08 / MWh (EUR 0.059 / kWh)
- **Weighted Electricity (Spot + Standard Grid Fees):** EUR 114.52 / MWh (EUR 0.115 / kWh)

$$\text{Baseline Cost/Ton} = (3.422 \times 59.08) + (0.933 \times 114.52) = EUR 202.17 + EUR 106.85 = \mathbf{EUR 309.02 / \text{ton}}$$
$$\text{Total Annual Site Energy Baseline} = 309.02 \times 450,000 = \mathbf{EUR 139,059,000 / \text{year}}$$


## 2. Configuration System & Data Schema Architecture

The optimization framework is powered by strongly typed **Pydantic configuration models** and component-level **TOML specifications** (`data/components/*.toml`), eliminating silent fallbacks and enforcing strict data validation at load time.

### Configuration Schema Reference:

| Schema Model | Scope / Component | Key Parameters & Validation | Description |
|---|---|---|---|
| `FacilityProjectConfig` | Master Project | `project_name`, `optimization_mode`, `start_time`, `end_time`, `wacc` (0.07), `co2_tax_eur_per_ton` (85.0) | Top-level project definition & economic environment settings. |
| `FixedSizingConfig` | Operation Hub | `pv` (kWp), `bess` (kWh), `hthp` (kW_th), `tes` (kWh_th) | Fixed existing asset capacities enforced in dispatch optimization. |
| `VariableSizingConfig` | Decision Hub | `pv`, `bess`, `hthp`, `tes` -> `ComponentBounds(min_capacity, max_capacity, enabled)` | Min/Max sizing bounds for investment candidate assets. |
| `PVComponentConfig` | `pv.toml` | `capex_eur_per_kw` (800.0), `lifetime_years` (25), `max_capacity_kw` (25000.0) | Rooftop PV technical & economic specs. |
| `BESSComponentConfig` | `bess.toml` | `capex_eur_per_kwh` (350.0), `lifetime_years` (15), `charge_efficiency` (0.95), `discharge_efficiency` (0.95) | Battery storage technical & economic specs. |
| `CHPComponentConfig` | `chp.toml` | `electrical_efficiency` (0.40), `thermal_efficiency` (0.45), `capacity_el_kw` (40000.0) | Combined Heat and Power turbine specs. |
| `EBoilerComponentConfig` | `eboiler.toml` | `thermal_efficiency` (0.98), `capacity_th_kw` (30000.0) | Power-to-Heat electrode boiler specs. |
| `HTHPComponentConfig` | `hthp.toml` | `capex_eur_per_kw_th` (600.0), `cop` (2.8), `lifetime_years` (20), `max_capacity_kw_th` (40000.0) | Industrial High-Temperature Heat Pump specs. |
| `TESComponentConfig` | Thermal Storage | `capex_eur_per_kwh_th` (120.0), `lifetime_years` (25), `max_capacity_kwh_th` (100000.0) | Sensible thermal energy storage specs. |


In [2]:
# =============================================================================
# OPERATION HUB USER CONFIGURATION (Pydantic Schema Validated)
# =============================================================================

op_config = FacilityProjectConfig(
    project_name="current_facility_optimization",
    optimization_mode="operation",
    start_time="01/01/2025",  # Start date in DD/MM/YYYY format
    end_time="08/01/2025",    # End date in DD/MM/YYYY format
    fixed_components_sizing=FixedSizingConfig(
        pv=0.0,         # Existing PV installed (kWp)
        bess=0.0,       # Existing BESS capacity (kWh)
        hthp=15000.0,   # Existing HTHP thermal capacity (kW_th)
        tes=20000.0,    # Existing Thermal Storage (kWh_th)
    ),
    co2_tax_eur_per_ton=85.0,
    enable_sec19_protection=True,
)

print(f"Project Name: {op_config.project_name}")
print(f"Mode: {op_config.optimization_mode.upper()} | Period: {op_config.start_time} to {op_config.end_time}")


Project Name: current_facility_optimization
Mode: OPERATION | Period: 01/01/2025 to 08/01/2025


In [3]:
# Prepare 2025 Datasets and Slice Analysis Window
#m_path, s_path = prepare_data_files(2025) #->If data does not exist or if year data wants to be modified 
m_path = r"/Users/mrafiindrajaya/Desktop/Github Projects/RIZM_challenge_Rafi/data/market_data_2025.csv" 
s_path = r"/Users/mrafiindrajaya/Desktop/Github Projects/RIZM_challenge_Rafi/data/solar_data_duesseldorf_2025.csv"
df_market_full = pd.read_csv(m_path, index_col=0, parse_dates=True)
df_solar_full  = pd.read_csv(s_path, index_col=0, parse_dates=True)

# Instantiate Operation Hub PyPSA Model
hes_op = HenkelEnergySystem(config=op_config, df_market=df_market_full, df_solar=df_solar_full)
n_op = hes_op.build_energy_system()

# Solve Operation Hub PyPSA Model
meta_op = hes_op.solve()

print("\n--- OPERATION HUB OPTIMIZATION SUMMARY ---")
display(pd.DataFrame([meta_op]))

# Render Interactive PyPSA Multi-Carrier Dispatch Stack
fig_op_dispatch = plot_dispatch_stacks_interactive(meta_op, title="Operation Hub PyPSA Multi-Carrier Dispatch Stack")
fig_op_dispatch.show()


Index(['b_elec', 'b_gas', 'b_steam_ht', 'b_heat_lt', 'bess_bus', 'tes_bus'], dtype='object', name='name')
Index(['gas_chp', 'gas_boiler', 'electric_boiler', 'steam_to_heat_exchanger',
       'heat_pump', 'bess_charger', 'bess_discharger', 'tes_charger',
       'tes_discharger'],
      dtype='object', name='name')
Index(['bess', 'tes'], dtype='object', name='name')
/Users/mrafiindrajaya/Desktop/Github Projects/RIZM_challenge_Rafi/src/optimization_model.py:506: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize(solver_name=solver_name, extra_functionality=apply_c_rate_coupling)
Index(['b_elec', 'b_gas', 'b_steam_ht', 'b_heat_lt', 'bess_bus', 'tes_bus'], dtype='object', name='name')
Index(['gas_chp', 'gas_boiler', 'electric_boiler', 'steam_to_heat

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-humheoi6 has 6760 rows; 3042 cols; 10816 nonzeros
Coefficient ranges:
  Matrix  [4e-01, 3e+00]
  Cost    [9e-03, 1e-01]
  Bound   [0e+00, 0e+00]
  RHS     [5e+03, 1e+06]
Presolving model
676 rows, 1705 cols, 2550 nonzeros 0s
338 rows, 1191 cols, 1698 nonzeros 0s
Dependent equations search running on 331 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
331 rows, 839 cols, 1332 nonzeros 0s
Presolve reductions: rows 331(-6429); columns 839(-2203); nonzeros 1332(-9484) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0.0s
        644     2.7093556393e+06 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution

,status,total_cost_eur,opex_eur,capex_annualized_eur,elec_cost_eur,gas_cost_eur,emissions_t_co2,peak_grid_demand_kw,sec19_violation,optimal_capacities,network
0,ok,2.709356e+06,2.709356e+06,0.0,498640.126523,2.210716e+06,12425.87607,90867.346939,True,"{'pv_kwp': 0.0, 'bess_kwh': 0.0, 'hthp_kw_th':...",PyPSA Network 'Unnamed Network'


In [4]:
# =============================================================================
# DECISION HUB USER CONFIGURATION (Pydantic Schema Validated)
# =============================================================================

inv_config = FacilityProjectConfig(
    project_name="future_facility_decarbonization",
    optimization_mode="investment",
    start_time="01/01/2025",
    end_time="08/01/2025",
    variable_components_sizing=VariableSizingConfig(),  # Uses default investment candidate bounds
    co2_tax_eur_per_ton=85.0,
    wacc=0.07,
)

print(f"Project Name: {inv_config.project_name}")
print(f"Mode: {inv_config.optimization_mode.upper()} | Period: {inv_config.start_time} to {inv_config.end_time}")


Project Name: future_facility_decarbonization
Mode: INVESTMENT | Period: 01/01/2025 to 08/01/2025


In [5]:
# Instantiate Decision Hub Investment Model
hes_inv = HenkelEnergySystem(config=inv_config, df_market=df_market_full, df_solar=df_solar_full)
n_inv = hes_inv.build_energy_system()

# Solve Decision Hub Investment Model
meta_inv = hes_inv.solve()

print("\n--- DECISION HUB OPTIMIZATION SUMMARY ---")
display(pd.DataFrame([meta_inv]))

# 1. Render PyPSA Optimal Asset Sizing Results Table
df_sizing = create_pypsa_asset_sizing_table(meta_inv)
print("\n--- OPTIMAL INVESTMENT ASSET SIZING ---")
display(df_sizing)

# 2. Render Financial & Technical Scenario Comparison Table
df_summary = create_summary_dataframe({"Operation Hub": meta_op, "Decision Hub": meta_inv})
print("\n--- EXECUTIVE SCENARIO COMPARISON SUMMARY TABLE ---")
display(df_summary)

# 3. Render Asset Financial Economics Bar Chart
fig_econ = plot_asset_economics_interactive(meta_inv, title="Decision Hub Asset Financial Economics")
fig_econ.show()


Index(['b_elec', 'b_gas', 'b_steam_ht', 'b_heat_lt', 'bess_bus', 'tes_bus'], dtype='object', name='name')
Index(['gas_chp', 'gas_boiler', 'electric_boiler', 'steam_to_heat_exchanger',
       'heat_pump', 'bess_charger', 'bess_discharger', 'tes_charger',
       'tes_discharger'],
      dtype='object', name='name')
Index(['bess', 'tes'], dtype='object', name='name')
/Users/mrafiindrajaya/Desktop/Github Projects/RIZM_challenge_Rafi/src/optimization_model.py:506: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize(solver_name=solver_name, extra_functionality=apply_c_rate_coupling)
Index(['b_elec', 'b_gas', 'b_steam_ht', 'b_heat_lt', 'bess_bus', 'tes_bus'], dtype='object', name='name')
Index(['gas_chp', 'gas_boiler', 'electric_boiler', 'steam_to_heat

Running HiGHS 1.15.1 (git hash: 04024d7): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
LP linopy-problem-eo0bcayo has 6780 rows; 3050 cols; 12085 nonzeros
Coefficient ranges:
  Matrix  [9e-04, 3e+00]
  Cost    [5e-01, 2e+02]
  Bound   [0e+00, 0e+00]
  RHS     [1e+04, 1e+06]
Presolving model
2428 rows, 2770 cols, 7284 nonzeros 0s
2090 rows, 2263 cols, 6439 nonzeros 0s
Dependent equations search running on 845 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
2090 rows, 2263 cols, 6439 nonzeros 0s
Presolve reductions: rows 2090(-4690); columns 2263(-787); nonzeros 6439(-5646) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 507(5.529e+07) 0.0s
       1550     1.2979394632e+08 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP fro

,status,total_cost_eur,opex_eur,capex_annualized_eur,elec_cost_eur,gas_cost_eur,emissions_t_co2,peak_grid_demand_kw,sec19_violation,optimal_capacities,network
0,ok,1.297939e+08,1.271285e+08,2.665430e+06,2.865717e+07,9.847135e+07,12057.84443,99795.918367,True,"{'pv_kwp': -0.0, 'bess_kwh': -0.0, 'hthp_kw_th...",PyPSA Network 'Unnamed Network'



--- OPTIMAL INVESTMENT ASSET SIZING ---


,Asset Component,Optimal Sizing Capacity,Unit
0,GRID_GAS,"1,000,000.00",kWp
1,STEAM_DUMP,"1,000,000.00",kWp
2,HEAT_DUMP,"1,000,000.00",kWp
3,GAS_CHP,"75,000.00",kW_th
4,GAS_BOILER,"195,652.17",kW_th
5,ELECTRIC_BOILER,"25,510.20",kW_th
6,HEAT_PUMP,"14,285.71",kW_th



--- EXECUTIVE SCENARIO COMPARISON SUMMARY TABLE ---


,Total Cost (EUR),Cost per Ton (EUR/ton),OPEX (EUR),Annualized CAPEX (EUR),Emissions (tCO2),Peak Grid Demand (MW),Sec19 Compliant
Scenario,,,,,,,
Operation Hub,2.709356e+06,6.020790,2.709356e+06,0.000000e+00,12425.87607,90.867347,False
Decision Hub,1.297939e+08,288.430992,1.271285e+08,2.665430e+06,12057.84443,99.795918,False


## 5. Strategic On-Site Protocol for Henkel Düsseldorf

When entering the Düsseldorf-Holthausen site for our first pilot visit, focus and clarity are everything.

### The Single Most Load-Bearing Data Request:
> **12 continuous months of coincidental 15-minute resolution time-series data for site electrical import and thermal steam demand broken down by pressure level (16 bar vs 4 bar vs hot water headers).**
>
> *Why this request?* Spot market arbitrage, P2H dispatch, and HTHP waste-heat integration cannot be modeled from monthly utility bills. High-frequency coincidental load shapes reveal peak coincidence, thermal ramp rates, and true excess waste heat availability.

### The Single Most Load-Bearing Stakeholder (30-Minute Agenda):
> **Head of On-Site Energy Utilities & Infrastructure (*Leiter Energieversorgung Holthausen*)**
>
> **30-Minute Value Proposition Agenda:**
> 1. **Minutes 0-5:** Present baseline EUR/ton energy cost breakdown and sec 19 StromNEV grid fee discount protection protocol.
> 2. **Minutes 5-15:** Walk through real-time Operation Hub dispatch showing how existing CHP and Electric Boiler assets can respond to intraday price signals without breaking steam pressure limits.
> 3. **Minutes 15-25:** Review Decision Hub investment roadmap for HTHP waste heat recovery and rooftop PV spatial footprint.
> 4. **Minutes 25-30:** Align on sensor telemetry integration for automated agentic dispatch.
